In [ ]:
#選 GPU & 基本檢查
import torch, subprocess, sys, platform, os, json
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
GPU_state=torch.cuda.is_available()
print("CUDA available:", GPU_state)
if not GPU_state:
  print("please select GPU")
!nvidia-smi -L || true

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cpu
CUDA available: False
please select GPU
/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# === 掛載 Google Drive ===
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# === 匯入必要模組 ===
from pathlib import Path
import os, shutil

# === 設定在雲端硬碟中的專案資料夾 ===
#   你的專案在 Google Drive 裡的路徑如下：
#   MyDrive/GGWP_US_LLTL/
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJ_DRIVE = DRIVE_ROOT / "GGWP_US_LLTL"   # 主專案資料夾
COLAB_ROOT = Path("/content/GGWP_US_LLTL") # Colab 本地端的虛擬資料夾

# === 確保每個子資料夾都存在（若不存在會自動建立） ===
for folder in ["data", "src", "data_cleaning"]:
    (PROJ_DRIVE / folder).mkdir(parents=True, exist_ok=True)

# === symlink 區塊（可視需要開啟） ===
# symlink（符號連結）能讓 Colab 的 /content/GGWP_US_LLTL
# 指向 Google Drive 的 GGWP_US_LLTL。
# 這樣你的程式可以直接使用相對路徑，例如：
#   ./data 對應到 Google Drive 的 data/
#   ./checkpoints 對應到 Google Drive 的 checkpoints/
# 因此不用每次都 copy 檔案。
#
# 如果不想用 symlink，可以註解掉這整段。

def link(src, dst):
    """建立符號連結（若目標已存在則自動移除重建）"""
    src, dst = Path(src), Path(dst)
    if dst.exists() or dst.is_symlink():
        if dst.is_dir() and not dst.is_symlink():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    os.symlink(src, dst, target_is_directory=True)

# 建立 Colab 對應的本地目錄
COLAB_ROOT.mkdir(parents=True, exist_ok=True)

#使用 symlink
# link(PROJ_DRIVE / "data",        COLAB_ROOT / "data")
# link(PROJ_DRIVE / "src",         COLAB_ROOT / "src")
# link(PROJ_DRIVE / "data_cleaning",         COLAB_ROOT / "data_cleaning")
link(PROJ_DRIVE          ,         COLAB_ROOT)
# 切換工作目錄到 src，方便執行 main.py
os.chdir(PROJ_DRIVE / "src")
print("✅ 專案環境初始化完成")
print("目前工作目錄：", os.getcwd())

# 注意：是否接續訓練 (resume_from_checkpoint)、要用哪個版本的清洗資料
# (data_version_override) 等設定，統一寫在 src/hyperparameters.py 裡，
# 不再透過這裡的環境變數控制。執行前請直接去該檔案確認/修改設定。

Mounted at /content/drive
✅ 專案環境初始化完成
目前工作目錄： /content/drive/MyDrive/GGWP_US_LLTL/src


In [ ]:
os.chdir(PROJ_DRIVE / "src")
!python preprocess.py


[Auto Resume] 偵測到 checkpoint：/content/GGWP_US_LLTL/data/Train_ckpt/2026-07-11 15-44-20/Epoch=75_ValLoss=7.4364.pth
[Ill-data log] 新建 v1：/content/GGWP_US_LLTL/data_cleaning/ill_conditioned_data_v1.csv
[Data Version Override] 訓練資料強制使用 cleaned_data_v0.pickle（本次 log 版本號 v1 跟資料版本號可能不同，這是預期行為——log 只是為了記錄「用修正後 loss 訓練 v0 時發現的異常樣本」，清洗時要對應套用到 v0.pickle，不是套用到自動偵測的最新版本）
[Resume] 2018 已有checkpoint，跳過重算
[Resume] 2019 已有checkpoint，跳過重算
[Resume] 2020 已有checkpoint，跳過重算
[Resume] 2021 已有checkpoint，跳過重算
[Resume] 2022 已有checkpoint，跳過重算
[Resume] 2023 已有checkpoint，跳過重算
整體進度: 100% 6/6 [05:45<00:00, 57.58s/year]

全部完成，共 5134879 筆樣本，已存到：/content/GGWP_US_LLTL/data/preprocess_data/new_preprocess_data/preprocess_data_2018_10_to_2023_10.pickle


In [ ]:
os.chdir(PROJ_DRIVE / "data_cleaning")
!python check_full_data_integrity.py

[Auto Resume] 偵測到 checkpoint：/content/GGWP_US_LLTL/data/Train_ckpt/2026-08-09 06-13-54/Epoch=69_ValLoss=12.0957.pth
[Ill-data log] 新建 v2：/content/GGWP_US_LLTL/data_cleaning/ill_conditioned_data_v2.csv
[Data Version Override] 訓練資料強制使用 cleaned_data_v0.pickle（本次 log 版本號 v2 跟資料版本號可能不同，這是預期行為——log 只是為了記錄「用修正後 loss 訓練 v0 時發現的異常樣本」，清洗時要對應套用到 v0.pickle，不是套用到自動偵測的最新版本）
共 1247 個檔案，開始掃描...
掃描完整度: 100% 1247/1247 [02:09<00:00,  9.64day/s]

完整度檢查摘要

[A] 交易日曆缺口（比對來源：純平日推算（無法排除國定假日，缺口清單需人工用行事曆確認））
  缺少 57 個應有交易日的檔案
  範例: ['2018-11-22', '2018-11-23', '2018-12-05', '2018-12-24', '2018-12-25', '2019-01-01', '2019-01-21', '2019-02-18', '2019-04-19', '2019-05-27']

[A] 列數不是標準值(390)的檔案數: 0

[A] 欄位（股票）數量趨勢: 最早 457 檔 -> 最新 503 檔

[B] 有「中間斷層」的股票數: 7
 ticker first_date  last_date  n_gap_days_in_middle                                            example_gap_dates
 WTW.OQ 2018-10-19 2023-10-18                   681 [2019-04-22, 2019-04-23, 2019-04-24, 2019-04-25, 2019-04-26]
AXON.OQ 2018-10-19 2023-10-18          

In [ ]:
os.chdir(PROJ_DRIVE / "create_formationtable")
#!python create_formationtable.py         # 跑非平行版
!python multiprocessing/Main_Check_parallel.py  # 跑平行版
#!python verify_check_table.py           # 跑驗證程式
os.chdir(PROJ_DRIVE / "src")


Streaming output truncated to the last 5000 lines.
  [配對失敗] ROK.N-TXT.N：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] RF.N-TRGP.N：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] RL.N-ULTA.OQ：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] RJF.N-SPGI.N：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] QRVO.OQ-WST.N：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] ROK.N-TYL.N：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] RF.N-TRMB.OQ：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] REGN.OQ-SCHW.N：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] RL.N-UNH.N：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] QCOM.OQ-TPR.N：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] RJF.N-SRE.N：IndexError('index 0 is out of bounds for axis 0 with size 0')
  [配對失敗] QRVO.OQ-WY.N：Index

In [ ]:
os.chdir(PROJ_DRIVE / "src")
!python main.py
# os.chdir(PROJ_DRIVE / "data_cleaning")
# !python clean_data.py

NameError: name 'os' is not defined

In [ ]:
# import time
# time.sleep(30)  # 暫停 30 秒


os.chdir(PROJ_DRIVE / "data_cleaning")
!python clean_data.py

[Auto Resume] 偵測到 checkpoint：/content/GGWP_US_LLTL/data/Train_ckpt/2026-07-11 15-44-20/Epoch=75_ValLoss=7.4364.pth
[Ill-data log] 新建 v1：/content/GGWP_US_LLTL/data_cleaning/ill_conditioned_data_v1.csv
[Data Version Override] 訓練資料強制使用 cleaned_data_v0.pickle（本次 log 版本號 v1 跟資料版本號可能不同，這是預期行為——log 只是為了記錄「用修正後 loss 訓練 v0 時發現的異常樣本」，清洗時要對應套用到 v0.pickle，不是套用到自動偵測的最新版本）
[Data Version Override] 強制清洗 v0（忽略自動偵測最新版本的邏輯）
📦 讀取原始資料版本：cleaned_data_v0.pickle
📝 參考異常 ID 紀錄（相關係數矩陣接近奇異）：ill_conditioned_data_v0.csv
📝 參考異常 ID 紀錄（sigma 崩塌）：sigma_collapse_data_v0.csv
🚀 清洗後將儲存為：cleaned_data_v1.pickle

🚀 開始執行資料清洗流程
✅ 成功載入 686415 筆異常 ID（相關係數矩陣接近奇異）
ℹ️ 找不到 sigma 崩塌 的紀錄檔（sigma_collapse_data_v0.csv），略過。
正在讀取 cleaned_data_v0.pickle...
⚠️ 原始資料無 Data_ID 欄位，自動將 index 設為 Data_ID
正在儲存至 cleaned_data_v1.pickle...

📊 清洗結果統計
  - 原始總筆數: 8,775,682
  - 剔除異常數: 84,505
  - 剩餘淨筆數: 8,691,177
  - 剔除百分比: 0.9629%


In [ ]:
time.sleep(30)  # 暫停 30 秒

os.chdir(PROJ_DRIVE / "src")
!python main.py
os.chdir(PROJ_DRIVE / "data_cleaning")
!python clean_data.py

In [ ]:
time.sleep(30)  # 暫停 30 秒

os.chdir(PROJ_DRIVE / "src")
!python main.py
os.chdir(PROJ_DRIVE / "data_cleaning")
!python clean_data.py

outlier儲存版本：v3，路徑：/content/GGWP_US_LLTL/data_cleaning/ill_conditioned_data_v3.csv
載入清洗後的乾淨資料...
讀取最新版本：v3，路徑：/content/GGWP_US_LLTL/data_cleaning/cleaned_data_v3.pickle
Training data date:
    From 20181019 to 20201016
Validation data date:
    From 20201019 to 20211018
Testing data date:
    From 20211019 to 20211231
All data length       : 913391
Train data length     : 592815
Validation data length: 206648
Test data length      : 113928
Train data shape: (592815, 3, 150)
Valid data shape: (206648, 3, 150)
Test  data shape: (113928, 3, 150)
(Test) Epoch 0, lr:0.00010000: 100%|████████████████████| 446/446 [00:05<00:00, 77.65it/s, loss=7.1]
Train set      loss : 5.432534
Validation set loss : 6.925638
Test set loss       : 6.965631
save ckpt to /content/GGWP_US_LLTL/data/Train_ckpt/2026-03-09 11-34-52/Epoch=1_ValLoss=6.9256.pth
Epoch [1/1000], Loss: 5.4325
save ckpt to /content/GGWP_US_LLTL/data/Train_ckpt/2026-03-09 11-34-52/BestEpoch=1_ValLoss=6.9256.pth
(Test) Epoch 1, lr:0.00010000

In [ ]:
time.sleep(30)  # 暫停 30 秒

os.chdir(PROJ_DRIVE / "src")
!python main.py
os.chdir(PROJ_DRIVE / "data_cleaning")
!python clean_data.py

In [ ]:
time.sleep(30)  # 暫停 30 秒

os.chdir(PROJ_DRIVE / "src")
!python main.py
os.chdir(PROJ_DRIVE / "data_cleaning")
!python clean_data.py

In [ ]:
os.chdir(PROJ_DRIVE / "data_cleaning")
!python clean_data.py

📦 讀取原始資料版本：cleaned_data_v0.pickle
📝 參考異常 ID 紀錄：ill_conditioned_data_v0.csv
🚀 清洗後將儲存為：cleaned_data_v1.pickle

🚀 開始執行資料清洗流程
✅ 成功載入 1022983 筆異常 ID
正在讀取 cleaned_data_v0.pickle...
⚠️ 原始資料無 Data_ID 欄位，自動將 index 設為 Data_ID
正在儲存至 cleaned_data_v1.pickle...

📊 清洗結果統計
  - 原始總筆數: 8,775,682
  - 剔除異常數: 92,163
  - 剩餘淨筆數: 8,683,519
  - 剔除百分比: 1.0502%


In [ ]:
import time
time.sleep(60)  # 暫停 30 秒

# 在你的所有程式碼執行完畢後，加入以下程式碼
try:
    from google.colab import runtime
    print("程式執行完畢，即將中斷 Colab 連線並釋放資源。")
    runtime.unassign()
except ImportError:
    print("非 Colab 環境，無需中斷連線。")


In [ ]:
import time, sys, os
import torch
import numpy as np

# 確保可以 import 你的模組
os.chdir('/content/drive/MyDrive/GGWP_US_LLTL/src')
sys.path.insert(0, '/content/drive/MyDrive/GGWP_US_LLTL/src')

from hyperparameters import args
import model as model_module

# ── 用隨機假資料測試，不需要載入幾GB的真實資料 ──
N = 5000
fake_data   = np.random.randn(N, 3, 150).astype(np.float32)
fake_labels = np.random.randn(N, 6).astype(np.float32)

PT_model = model_module.GGPTS(args).to(args.device)

# ── 測試1：純 DataLoader 速度（不跑 forward）──
loader = PT_model.load_dataloader(fake_data, fake_labels)
start = time.time()
for i, (data, label) in enumerate(loader):
    data = data.to(args.device)
    if i == 50: break
t_load = time.time() - start
print(f"純讀取 50 batches：{t_load:.2f}s  ({t_load/50*1000:.1f} ms/batch)")

# ── 測試2：加上 forward pass ──
loader = PT_model.load_dataloader(fake_data, fake_labels)
start = time.time()
for i, (data, label) in enumerate(loader):
    data = data.to(args.device)
    with torch.no_grad():
        out = PT_model(data)
    if i == 50: break
t_forward = time.time() - start
print(f"含 forward 50 batches：{t_forward:.2f}s  ({t_forward/50*1000:.1f} ms/batch)")

# ── 結論判斷 ──
ratio = t_forward / t_load
print(f"\nForward/Load 比值：{ratio:.1f}x")
if ratio > 3:
    print("→ 瓶頸在模型計算（GPU），優化 DataLoader 幫助有限")
else:
    print("→ 瓶頸在資料讀取，增加 num_workers 有效")

[Auto Resume] 偵測到 checkpoint：/content/GGWP_US_LLTL/data/Train_ckpt/2026-04-02 07-11-01/Epoch=3_ValLoss=10.7101.pth
[Ill-data log] 接續寫入 v0：/content/GGWP_US_LLTL/data_cleaning/ill_conditioned_data_v0.csv
純讀取 50 batches：0.12s  (2.4 ms/batch)
含 forward 50 batches：1.18s  (23.5 ms/batch)

Forward/Load 比值：9.9x
→ 瓶頸在模型計算（GPU），優化 DataLoader 幫助有限
